In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Ton modèle choisi
model_new = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

# Ancien modèle pour comparaison
model_old = SentenceTransformer("all-MiniLM-L6-v2")

# Questions de test
questions = [
    ("Comment intégrer l'ENSA Fès ?", "L'admission se fait via le CNC pour les bacheliers scientifiques."),
    ("Quels clubs existe à l'école ?", "L'ENSA Fès dispose de clubs : Informatique, Robotique, Sportif."),
    ("Durée de la formation ?", "La formation dure 5 ans : 2 ans CPGE + 3 ans cycle ingénieur."),
]

print("="*65)
print(f"{'Question':<40} {'Ancien':>10} {'Nouveau':>10}")
print("="*65)

for question, reponse in questions:
    # Similarité cosinus avec ancien modèle
    e1 = model_old.encode([question, reponse])
    sim_old = float(np.dot(e1[0], e1[1]) / (np.linalg.norm(e1[0]) * np.linalg.norm(e1[1])))

    # Similarité cosinus avec nouveau modèle
    e2 = model_new.encode([question, reponse])
    sim_new = float(np.dot(e2[0], e2[1]) / (np.linalg.norm(e2[0]) * np.linalg.norm(e2[1])))

    meilleur = "NEW" if sim_new > sim_old else "OLD"
    print(f"{question:<40} {sim_old:>10.4f} {sim_new:>10.4f}  <- {meilleur}")

print("="*65)
print("Plus la similarité est haute, meilleur est le modèle pour ce cas.")

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Question                                     Ancien    Nouveau
Comment intégrer l'ENSA Fès ?                0.4526     0.2817  <- OLD
Quels clubs existe à l'école ?               0.6266     0.6217  <- OLD
Durée de la formation ?                      0.5523     0.6313  <- NEW
Plus la similarité est haute, meilleur est le modèle pour ce cas.


In [2]:
import faiss, json, numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)

passages_alignes = [
    item["question"] + " " + item["answer"]
    for item in ensa_data
]

print(f"Encodage de {len(passages_alignes)} passages...")
embeddings = model.encode(passages_alignes, show_progress_bar=True)

# IMPORTANT : multi-qa-MiniLM utilise similarité cosinus
# donc on normalise les vecteurs avant d'utiliser IndexFlatIP (produit interne)
# car cosinus(a,b) = produit_interne(a/|a|, b/|b|)
faiss.normalize_L2(embeddings.astype(np.float32))

dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)  # IP = Inner Product (cosinus après normalisation)
index.add(embeddings.astype(np.float32))

faiss.write_index(index, "ensa_faiss_v2.index")
print(f"Nouvel index sauvegardé : ensa_faiss_v2.index")
print(f"Vecteurs indexés : {index.ntotal}")
print(f"Dimension : {dimension}")

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Encodage de 42 passages...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Nouvel index sauvegardé : ensa_faiss_v2.index
Vecteurs indexés : 42
Dimension : 384


In [3]:
def retrieve_v2(query, top_k=3):
    """
    Retriever adapté à multi-qa-MiniLM-L6-cos-v1.
    
    Différence clé avec la v1 :
    - v1 (all-MiniLM) utilisait distance L2  → plus petit = meilleur
    - v2 (multi-qa)   utilise similarité cosinus → plus GRAND = meilleur
    
    Seuil : > 0.5 excellent, 0.3-0.5 bon, < 0.3 hors KB
    """
    query_vec = model.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)  # normalisation obligatoire pour cosinus

    scores, indices = index.search(query_vec, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(ensa_data):
            results.append({
                "rank": i + 1,
                "score_cosinus": float(scores[0][i]),  # entre 0 et 1
                "answer": ensa_data[idx]["answer"],
                "question": ensa_data[idx]["question"],
                "category": ensa_data[idx]["category"]
            })
    return results

# TEST
print("=== TEST RETRIEVER V2 ===")
for q in ["Comment intégrer l'ENSA ?", "Clubs disponibles ?", "Où est l'ENSA ?"]:
    r = retrieve_v2(q)
    print(f"\nQ: {q}")
    for p in r:
        print(f"  Rank {p['rank']} | cosinus={p['score_cosinus']:.4f} | {p['question'][:55]}")

=== TEST RETRIEVER V2 ===

Q: Comment intégrer l'ENSA ?
  Rank 1 | cosinus=0.4363 | Quel est le contact de l'administration de l'ENSA Fès ?
  Rank 2 | cosinus=0.4344 | Quels sont les enseignants du département Mathématiques
  Rank 3 | cosinus=0.4075 | Combien d'années dure la formation à l'ENSA Fès ?

Q: Clubs disponibles ?
  Rank 1 | cosinus=0.7384 | Quels sont les clubs disponibles à l'ENSA Fès ?
  Rank 2 | cosinus=0.5179 | Comment rejoindre un club à l'ENSA Fès ?
  Rank 3 | cosinus=0.1937 | L'ENSA Fès propose-t-elle des formations continues ?

Q: Où est l'ENSA ?
  Rank 1 | cosinus=0.5561 | Quel est le contact de l'administration de l'ENSA Fès ?
  Rank 2 | cosinus=0.5334 | Quel est le calendrier académique de l'ENSA Fès ?
  Rank 3 | cosinus=0.5102 | Où se trouve l'ENSA Fès ?


In [7]:
SEUIL_COSINUS = 0.45  # En dessous de 0.35 → question hors KB

def calculer_confiance_v2(score_cosinus, score_qa):
    """
    Formule adaptée au score cosinus (contrairement à FAISS L2).
    
    score_cosinus : entre 0 et 1 (plus grand = meilleur)
    score_qa      : entre 0 et 1 (confiance RoBERTa)
    
    Pas besoin d'inverser comme avec L2 — directement utilisable.
    Formule : 60% cosinus + 40% QA
    """
    if score_cosinus < SEUIL_COSINUS:
        return 0  # Hors KB → confiance nulle

    confiance = round((0.6 * score_cosinus + 0.4 * score_qa) * 100)
    return min(confiance, 99)  # Jamais 100% par humilité

# TEST
print("=== TEST SCORES ===")
print(f"cosinus=0.85, QA=0.92 → {calculer_confiance_v2(0.85, 0.92)}%  (attendu: ~88%)")
print(f"cosinus=0.55, QA=0.70 → {calculer_confiance_v2(0.55, 0.70)}%  (attendu: ~61%)")
print(f"cosinus=0.30, QA=0.80 → {calculer_confiance_v2(0.30, 0.80)}%  (attendu: 0% hors KB)")

=== TEST SCORES ===
cosinus=0.85, QA=0.92 → 88%  (attendu: ~88%)
cosinus=0.55, QA=0.70 → 61%  (attendu: ~61%)
cosinus=0.30, QA=0.80 → 0%  (attendu: 0% hors KB)


In [8]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# Charger RoBERTa manuellement (compatible toutes versions transformers)
model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name)

def read_answer(question, context):
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = qa_model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )
    score = float(torch.softmax(outputs.start_logits, dim=1).max())
    return answer, score


def chatbot_ensa_v3(query, top_k=3):
    """
    Pipeline RAG v3 — corrections appliquées :
    - Seuil cosinus relevé à 0.45
    - RoBERTa reçoit seulement le meilleur passage (pas les 3 combinés)
    - Score confiance recalibré
    """
    # ── ÉTAPE 1 : Retrieval ───────────────────────────────────
    passages = retrieve_v2(query, top_k)
    if not passages:
        return {"reponse": "Erreur interne.", "confiance": 0}

    meilleur = passages[0]

    # ── ÉTAPE 2 : Vérification seuil ─────────────────────────
    if meilleur["score_cosinus"] < SEUIL_COSINUS:
        return {
            "reponse": "Information non disponible dans ma base ENSA. Contactez contact@ensa-fes.ma",
            "confiance": 0,
            "source": None,
            "category": "hors_kb"
        }

    # ── ÉTAPE 3 : RoBERTa sur le MEILLEUR passage uniquement ──
    # Correction clé : un seul passage = contexte plus propre = score QA plus élevé
    contexte = meilleur["answer"]
    reponse, score_qa = read_answer(query, contexte)

    # ── ÉTAPE 4 : Score confiance recalibré ───────────────────
    confiance = calculer_confiance_v2(meilleur["score_cosinus"], score_qa)

    return {
        "reponse": reponse,
        "source": meilleur["question"],
        "confiance": confiance,
        "category": meilleur["category"],
        "score_cosinus": round(meilleur["score_cosinus"], 4),
        "score_qa": round(score_qa, 4),
        "passage": meilleur["answer"]
    }

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
import pandas as pd

questions_benchmark = [
    "Comment intégrer l'ENSA Fès ?",
    "Quels clubs y a-t-il à l'école ?",
    "Quelle est la durée de la formation ?",
    "Y a-t-il des stages obligatoires ?",
    "Où se trouve l'ENSA Fès ?",
    "Quelles sont les filières disponibles ?",
    "Quels sont les frais de scolarité ?",
    "Quel est le meilleur restaurant près de l'ENSA ?",
    "Comment apprendre le machine learning ?",
    "Quelle est la météo à Fès aujourd'hui ?",
]

resultats = []
print("Benchmark v3 en cours...\n")

for q in questions_benchmark:
    res = chatbot_ensa_v3(q)
    c = res["confiance"]
    badge = "VERT  " if c > 70 else "ORANGE" if c > 40 else "ROUGE "

    resultats.append({
        "Question": q[:42] + "...",
        "Réponse": res["reponse"][:45] + "...",
        "Confiance": f"{c}%",
        "Badge": badge,
        "Cos.": res.get("score_cosinus", "N/A"),
        "QA": res.get("score_qa", "N/A"),
        "Catégorie": res.get("category", "N/A")
    })

df = pd.DataFrame(resultats)
print(df.to_string(index=False))

in_kb  = [r for r in resultats[:7] if r["Badge"] != "ROUGE "]
hors_kb = [r for r in resultats[7:] if r["Catégorie"] == "hors_kb"]

print(f"\nRésumé v3 :")
print(f"  Questions KB  avec confiance >40% : {len(in_kb)}/7")
print(f"  Questions hors KB bien détectées  : {len(hors_kb)}/3")
print(f"  Questions VERT (>70%)             : {len([r for r in resultats if r['Badge']=='VERT  '])}")

Benchmark v3 en cours...

                                     Question                                          Réponse Confiance  Badge    Cos.      QA Catégorie
             Comment intégrer l'ENSA Fès ?...                                         email...       40% ROUGE   0.5803  0.1312  contacts
          Quels clubs y a-t-il à l'école ?...  nombreux clubs étudiants couvrant des domain...       52% ORANGE  0.6582  0.3206     clubs
     Quelle est la durée de la formation ?...                                              ...       49% ORANGE  0.4986  0.4657  filieres
        Y a-t-il des stages obligatoires ?...                                              ...       57% ORANGE  0.7625  0.2855    stages
                 Où se trouve l'ENSA Fès ?...                                              ...       58% ORANGE  0.6298   0.502  contacts
   Quelles sont les filières disponibles ?... Quelles sont les filières disponibles?L'ENSA ...       44% ORANGE  0.4896  0.3722  filieres
       Q